In [ ]:
import json

def read_json(filepath):
    return json.load(open(filepath,'r')) 

filepath = '/data/home/zhangjs/disk/project/verl-agent/data_pipelines/gamefiles/gamefiles_train.json'

In [ ]:
total_train_data = read_json(filepath=filepath)

gamefiles = list(total_train_data.values())

In [ ]:
synthestic = '/data/home/zhangjs/disk/project/verl-agent/data_pipelines/datas/parallel_vanilla/win.json'

synthestic_data = read_json(synthestic)
synthestic_gamefiles = [elem['game_file'] for elem in synthestic_data]

In [ ]:
left_gamefiles = list(set(gamefiles) - set(synthestic_gamefiles))

len(left_gamefiles) 

### Get the GT

In [ ]:
gt_data = read_json('/data/home/zhangjs/disk/project/verl-agent/data_pipelines/datas/ground_truth/alfworld_train_3553.json')

gt_process = [elem['short_gt'] for elem in gt_data]

gt_dict = {}
for elem in gt_process:
    file_name = elem['game_file'][0]
    gt_dict[file_name] = {}
    gt_dict[file_name]['expert_actions'] = []
    gt_dict[file_name]['expert_observations'] = []
    for conv in elem['observations']:
        gt_dict[file_name]['expert_actions'].append(conv['plan'])
        gt_dict[file_name]['expert_observations'].append(conv['observation'])



In [ ]:
parquet_train_data = []
for file in left_gamefiles:
    parquet_train_data.append(
        {
            'answer': '',
            'data_source': 'alfworld',
            'prompt': [{'role': 'user', 'content': 'The prompt is dynamic obtained from envs'}],
            'ability': 'agent',
            'gamefile': file,
            'extra_info': {
                'split': 'train'
            },
            'expert_actions': gt_dict[file]['expert_actions'],
            'expert_observations':gt_dict[file]['expert_observations'],
        }
    )


In [ ]:
existing_500 = read_json('/data/home/zhangjs/disk/project/verl-agent/data_pipelines/gamefiles/dedu_parallel_train_data_500.json') 
extsing_file_name = set(existing_500.values())
len(extsing_file_name)

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

current_data = [elem for elem in parquet_train_data if elem['gamefile'] in extsing_file_name]
df = pd.DataFrame(current_data)

print(len(current_data))

In [ ]:
df.to_parquet('../verl_train_data/dedu_parallel_train_data_500_w_gt.parquet', index=False) 

In [ ]:
parquet_train_data[0] 

In [ ]:
game_files = {}
for idx,elem in enumerate(current_data):
    gamefile = elem['gamefile']
    game_files[str(idx)] = gamefile

with open('../gamefiles/dedu_parallel_train_data_w_500_gt.json','w') as f:
    json.dump(game_files,f,indent=4)

In [1]:
import pandas as pd
df = pd.read_parquet('/data/home/zhangjs/disk/project/verl-agent/data_pipelines/verl_train_data/dedu_parallel_train_data_500_w_gt.parquet')

len(df)

500

In [2]:
df.head(5)

,answer,data_source,prompt,ability,gamefile,extra_info,expert_actions,expert_observations
0,,alfworld,[{'content': 'The prompt is dynamic obtained f...,agent,/data/home/zhangjs/.cache/alfworld/json_2.1.1/...,{'split': 'train'},"[go to sidetable 1, use desklamp 1, take statu...",[You arrive at sidetable 1. On the sidetable 1...
1,,alfworld,[{'content': 'The prompt is dynamic obtained f...,agent,/data/home/zhangjs/.cache/alfworld/json_2.1.1/...,{'split': 'train'},"[go to dresser 1, use desklamp 1, take creditc...","[You arrive at dresser 1. On the dresser 1, yo..."
2,,alfworld,[{'content': 'The prompt is dynamic obtained f...,agent,/data/home/zhangjs/.cache/alfworld/json_2.1.1/...,{'split': 'train'},"[go to sinkbasin 1, take plate 1 from sinkbasi...",[You arrive at sinkbasin 1. On the sinkbasin 1...
3,,alfworld,[{'content': 'The prompt is dynamic obtained f...,agent,/data/home/zhangjs/.cache/alfworld/json_2.1.1/...,{'split': 'train'},"[go to bed 1, take cellphone 1 from bed 1, go ...","[You arrive at bed 1. On the bed 1, you see a ..."
4,,alfworld,[{'content': 'The prompt is dynamic obtained f...,agent,/data/home/zhangjs/.cache/alfworld/json_2.1.1/...,{'split': 'train'},"[go to sidetable 1, take remotecontrol 3 from ...",[You arrive at sidetable 1. On the sidetable 1...


In [ ]:
current_files = set([elem['gamefile'] for elem in parquet_train_data])

len(current_files & extsing_file_name)